In [5]:
#excel sheet with outliers

import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

BIDS_ROOT = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
DERIV     = BIDS_ROOT / "derivatives"
OUT_FILE  = DERIV / "outlier-analysis" / "exclusion_criteria_workbook.xlsx"

# ── Load ──────────────────────────────────────────────────────────────────────
def load(path):
    df = pd.read_csv(path, sep="\t")
    df.columns = df.columns.str.strip()
    if "subj_id" in df.columns:
        df = df.rename(columns={"subj_id": "VP"})
    df["phase"] = df["phase"].str.lower()
    return df

hrv_t    = load(DERIV/"hrv-analysis"/"task-BBSIG_per_phase_hrv-time.tsv")
hrv_f    = load(DERIV/"hrv-analysis"/"task-BBSIG_per_phase_hrv-freq.tsv")
hr_df    = load(DERIV/"hrv-analysis"/"task-BBSIG_per_phase_hr_nk.tsv")
bf_df    = load(DERIV/"bf-analysis"/"task-BBSIG_per_phase_bf.tsv")

for fname in ["task-BBSIG_eda-temp-bf_per_phase_final.tsv",
              "task-BBSIG_eda-temp-bf_per_phase_clean.tsv",
              "task-BBSIG_eda-temp-bf_per_phase.tsv"]:
    p = DERIV/"eda-analysis"/fname
    if p.exists():
        eda_temp = load(p)
        break

# ── EXACT column names (no guessing) ─────────────────────────────────────────
MEASURES = [
    # (sheet_label, df, exact_col, phys_low, phys_high, unit, description)
    ("RMSSD",       hrv_t,    "RMSSD",    5,    200,  "ms",
     "Physiological: 5–200 ms (Task Force 1996). sub-021 = NaN (signal loss)"),
    ("HFn",         hrv_f,    "HFn",      0,    1,    "0–1",
     "Physiological: 0–1 (normalised). Valid only if BF 9–24 br/min"),
    ("EDA",         eda_temp, "EDA_mean", 0.01, 60,   "µS",
     "Physiological: 0.01–60 µS. >200 µS = sensor saturation"),
    ("Temperature", eda_temp, "temp_mean",28,   40,   "°C",
     "Physiological: 28–40 °C. <28 °C = sensor detachment. Missing=temp_excluded=1"),
    ("BF",          bf_df,    "BF_bpm",   4,    40,   "br/min",
     "Valid for HFn: 9–24 br/min. Physiological range: 4–40 br/min"),
    ("HR",          hr_df,    "HR_mean",  40,   120,  "bpm",
     "Your criterion: 40–120 bpm (script 01). Task Force: 40–180 bpm"),
]

RSP_LOW, RSP_HIGH = 9, 24
PHASES = ["pre", "early", "late", "post"]

# BF lookup for RSP validity
bf_lookup = bf_df.set_index(["VP","ses_id","phase"])["BF_bpm"]

# ── Styles ────────────────────────────────────────────────────────────────────
FILL_RED    = PatternFill("solid", fgColor="FF9999")
FILL_ORANGE = PatternFill("solid", fgColor="FFCC99")
FILL_YELLOW = PatternFill("solid", fgColor="FFFF99")
FILL_GREEN  = PatternFill("solid", fgColor="CCFFCC")
FILL_GREY   = PatternFill("solid", fgColor="DDDDDD")
FILL_HEADER = PatternFill("solid", fgColor="2166AC")
FILL_LEGEND = PatternFill("solid", fgColor="EEF4FF")
FILL_EXCL   = PatternFill("solid", fgColor="FF6666")  # summary sheet excluded

FW = Font(bold=True, color="FFFFFF", name="Arial", size=9)
FN = Font(name="Arial", size=9)
FB = Font(name="Arial", size=9, bold=True)
thin = Side(style="thin", color="CCCCCC")
BRD  = Border(left=thin, right=thin, top=thin, bottom=thin)

COLUMNS = ["VP","ses_id","condition","phase","raw_value",
           "phys_flag","extremely_low","extremely_high",
           "flag_1SD","flag_2SD","flag_3SD","rsp_invalid",
           "n_flags","exclude_any","note"]
CW = {"VP":10,"ses_id":8,"condition":14,"phase":8,"raw_value":12,
      "phys_flag":11,"extremely_low":14,"extremely_high":15,
      "flag_1SD":9,"flag_2SD":9,"flag_3SD":9,"rsp_invalid":12,
      "n_flags":9,"exclude_any":12,"note":48}

def hdr(ws, row, col, val, fill=FILL_HEADER, font=FW, wrap=True):
    c = ws.cell(row=row, column=col, value=val)
    c.font = font; c.fill = fill; c.border = BRD
    c.alignment = Alignment(horizontal="center", vertical="center",
                            wrap_text=wrap)
    return c

def fmt(ws, row, fill, note_col=15):
    for ci in range(1, len(COLUMNS)+1):
        c = ws.cell(row=row, column=ci)
        c.font = FN; c.border = BRD
        c.alignment = Alignment(horizontal="center")
        # raw_value and note stay white for readability
        c.fill = PatternFill("solid", fgColor="FFFFFF") \
                 if ci in (5, note_col) else fill
        if ci == note_col:
            c.alignment = Alignment(horizontal="left", wrap_text=True)

# ── Per-measure sheets ────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# store (sheet_name, n_data_cols, data_start_row) for summary formulas
sheet_meta = {}

for label, src_df, val_col, phys_low, phys_high, unit, desc in MEASURES:
    ws = wb.create_sheet(title=label)

    # Description
    ws.append([f"{label} ({unit}) — {desc}"])
    ws.merge_cells(start_row=1, start_column=1, end_row=1,
                   end_column=len(COLUMNS))
    ws["A1"].font  = Font(bold=True, italic=True, name="Arial",
                          size=9, color="333333")
    ws["A1"].fill  = FILL_LEGEND
    ws.row_dimensions[1].height = 18

    # Header
    for ci, cn in enumerate(COLUMNS, 1):
        hdr(ws, 2, ci, cn)
    ws.row_dimensions[2].height = 30

    # Stats
    vals = pd.to_numeric(src_df[val_col], errors="coerce")
    gm, gs = vals.mean(), vals.std(ddof=1)

    data_start = 3
    participants_in_sheet = []

    sort_key = lambda s: s if s.name != "phase" \
               else s.map({p: i for i, p in enumerate(PHASES)})
    for _, row in src_df.sort_values(["VP","ses_id","phase"],
                                     key=sort_key).iterrows():
        vp   = str(row.get("VP",""))
        ses  = str(row.get("ses_id",""))
        cond = str(row.get("condition",""))
        ph   = str(row.get("phase",""))

        raw = row.get(val_col, np.nan)
        try:
            raw = float(raw)
            missing = np.isnan(raw)
        except:
            missing = True; raw = None

        if vp not in participants_in_sheet:
            participants_in_sheet.append(vp)

        if missing:
            ws.append([vp,ses,cond,ph,"","","","","","","","",
                       0,"","MISSING — excluded in source pipeline"])
            fmt(ws, ws.max_row, FILL_GREY)
            continue

        pf  = raw < phys_low or raw > phys_high
        el  = raw < gm - 3*gs
        eh  = raw > gm + 3*gs
        f1  = abs(raw - gm) > gs
        f2  = abs(raw - gm) > 2*gs
        f3  = abs(raw - gm) > 3*gs
        rsp = False
        if label in ("HFn","BF"):
            try:
                bv  = float(bf_lookup.loc[(vp, ses, ph)])
                rsp = bv < RSP_LOW or bv > RSP_HIGH
            except:
                pass

        nf   = sum([pf, el, eh, f1, f2, f3, rsp])
        excl = pf or f3 or rsp

        parts = []
        if pf:  parts.append(
            f"outside phys [{phys_low}–{phys_high}] (val={raw:.2f})")
        if el:  parts.append(f"extremely low <{gm-3*gs:.2f}")
        if eh:  parts.append(f"extremely high >{gm+3*gs:.2f}")
        if f3 and not (el or eh): parts.append("3SD outlier")
        elif f2 and not f3:       parts.append("2SD outlier")
        if rsp: parts.append(f"RSP invalid (BF out of 9–24 br/min)")
        note = "; ".join(parts) if parts else "OK"

        ws.append([vp, ses, cond, ph, round(raw, 4),
                   "YES" if pf else "", "YES" if el else "",
                   "YES" if eh else "", "YES" if f1 else "",
                   "YES" if f2 else "", "YES" if f3 else "",
                   "YES" if rsp else "", nf,
                   "YES" if excl else "", note])

        fill = (FILL_RED if excl else FILL_ORANGE if f2
                else FILL_YELLOW if f1 else FILL_GREEN)
        fmt(ws, ws.max_row, fill)

    ws.freeze_panes = "A3"
    for ci, cn in enumerate(COLUMNS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = CW.get(cn, 12)
    ws.row_dimensions[2].height = 30

    # Summary stats box top-right
    sc = len(COLUMNS) + 2
    for r, k, v in [
        (2,  "SUMMARY",          ""),
        (3,  "Group mean",       round(gm, 4)),
        (4,  "Group SD",         round(gs, 4)),
        (5,  "Mean − 1SD",       round(gm -   gs, 4)),
        (6,  "Mean + 1SD",       round(gm +   gs, 4)),
        (7,  "Mean − 2SD",       round(gm - 2*gs, 4)),
        (8,  "Mean + 2SD",       round(gm + 2*gs, 4)),
        (9,  "Mean − 3SD",       round(gm - 3*gs, 4)),
        (10, "Mean + 3SD",       round(gm + 3*gs, 4)),
        (11, "Phys low",         phys_low),
        (12, "Phys high",        phys_high),
        (13, "RSP valid low",    RSP_LOW),
        (14, "RSP valid high",   RSP_HIGH),
    ]:
        ws.cell(row=r, column=sc,   value=k).font = FB
        ws.cell(row=r, column=sc+1, value=v).font = FN
    ws.cell(row=2, column=sc).fill = FILL_HEADER
    ws.cell(row=2, column=sc).font = FW
    ws.column_dimensions[get_column_letter(sc)].width   = 15
    ws.column_dimensions[get_column_letter(sc+1)].width = 10

    data_end = ws.max_row
    sheet_meta[label] = {
        "data_start": data_start,
        "data_end":   data_end,
        "n_rows":     data_end - data_start + 1,
    }
    print(f"  Sheet '{label}': {data_end - data_start + 1} data rows "
          f"(gm={gm:.2f}, gs={gs:.2f})")

# ── SUMMARY SHEET — flags per participant ─────────────────────────────────────
# Collect all unique VPs across all source data
all_vps = sorted(set(
    list(hrv_t["VP"].unique()) +
    list(hrv_f["VP"].unique()) +
    list(eda_temp["VP"].unique()) +
    list(bf_df["VP"].unique()) +
    list(hr_df["VP"].unique())
))

measure_labels = [m[0] for m in MEASURES]

ws_sum = wb.create_sheet(title="FLAG_SUMMARY", index=0)

# Title
ws_sum.append(["FLAG SUMMARY — Flags per participant across all measures and sessions"])
ws_sum.merge_cells(start_row=1, start_column=1,
                   end_row=1, end_column=len(measure_labels)+4)
ws_sum["A1"].font  = Font(bold=True, name="Arial", size=10, color="FFFFFF")
ws_sum["A1"].fill  = FILL_HEADER
ws_sum["A1"].alignment = Alignment(horizontal="center")
ws_sum.row_dimensions[1].height = 20

# Sub-header explaining logic
ws_sum.append(["Counts = number of phase-cells flagged with exclude_any=YES "
               "(phys_flag OR flag_3SD OR rsp_invalid). "
               "Sort column N descending to find most-problematic participants."])
ws_sum.merge_cells(start_row=2, start_column=1,
                   end_row=2, end_column=len(measure_labels)+4)
ws_sum["A2"].font  = Font(italic=True, name="Arial", size=8, color="555555")
ws_sum["A2"].fill  = FILL_LEGEND
ws_sum.row_dimensions[2].height = 14

# Header row
sum_header = ["VP"] + measure_labels + ["TOTAL_FLAGS", "RANK", "VERDICT"]
for ci, val in enumerate(sum_header, 1):
    hdr(ws_sum, 3, ci, val)
ws_sum.row_dimensions[3].height = 28

# Data: compute flag counts from each measure sheet using COUNTIFS formula
# Formula: count rows in measure sheet where VP matches AND exclude_any="YES"
# Column positions in each measure sheet:
#   A=VP(1), N=exclude_any(14)
VP_COL      = "A"   # column A in each measure sheet
EXCL_COL    = "N"   # column N = exclude_any

data_row_start = 4

for row_i, vp in enumerate(all_vps, data_row_start):
    ws_sum.cell(row=row_i, column=1, value=vp).font = FB

    flag_cols = []
    for col_i, (label, *_) in enumerate(MEASURES, 2):
        meta      = sheet_meta[label]
        r_start   = meta["data_start"]
        r_end     = meta["data_end"]
        # COUNTIFS: VP in col A matches AND exclude_any in col N = "YES"
        formula   = (f"=COUNTIFS('{label}'!{VP_COL}{r_start}:"
                     f"'{label}'!{VP_COL}{r_end},A{row_i},"
                     f"'{label}'!{EXCL_COL}{r_start}:"
                     f"'{label}'!{EXCL_COL}{r_end},\"YES\")")
        c = ws_sum.cell(row=row_i, column=col_i, value=formula)
        c.font   = FN
        c.border = BRD
        c.alignment = Alignment(horizontal="center")
        flag_cols.append(get_column_letter(col_i))

    # TOTAL = SUM of all measure flag counts for this row
    sum_range = "+".join([f"{cl}{row_i}" for cl in flag_cols])
    total_col = len(MEASURES) + 2
    c_total   = ws_sum.cell(row=row_i, column=total_col,
                             value=f"={sum_range}")
    c_total.font   = FB
    c_total.border = BRD
    c_total.alignment = Alignment(horizontal="center")

    # Colour total cell by severity
    c_total.fill = FILL_RED if True else FILL_GREEN  # will be coloured after

    # RANK formula (rank by total descending among all participants)
    rank_col  = total_col + 1
    rank_range_start = data_row_start
    rank_range_end   = data_row_start + len(all_vps) - 1
    rank_total_col   = get_column_letter(total_col)
    c_rank = ws_sum.cell(
        row=row_i, column=rank_col,
        value=(f"=RANK({rank_total_col}{row_i},"
               f"${rank_total_col}${rank_range_start}:"
               f"${rank_total_col}${rank_range_end},1)")
    )
    c_rank.font   = FN
    c_rank.border = BRD
    c_rank.alignment = Alignment(horizontal="center")

    # VERDICT
    verdict_col = rank_col + 1
    c_verdict   = ws_sum.cell(
        row=row_i, column=verdict_col,
        value=f'=IF({rank_total_col}{row_i}=0,"CLEAN",'
              f'IF({rank_total_col}{row_i}<=2,"MONITOR",'
              f'IF({rank_total_col}{row_i}<=5,"INVESTIGATE","REVIEW")))'
    )
    c_verdict.font   = FN
    c_verdict.border = BRD
    c_verdict.alignment = Alignment(horizontal="center")

    # Row border and VP style
    ws_sum.cell(row=row_i, column=1).border = BRD
    ws_sum.cell(row=row_i, column=1).alignment = Alignment(horizontal="left")

# Column widths for summary sheet
ws_sum.column_dimensions["A"].width = 11
for col_i in range(2, len(MEASURES)+2):
    ws_sum.column_dimensions[get_column_letter(col_i)].width = 11
ws_sum.column_dimensions[get_column_letter(len(MEASURES)+2)].width = 13
ws_sum.column_dimensions[get_column_letter(len(MEASURES)+3)].width = 7
ws_sum.column_dimensions[get_column_letter(len(MEASURES)+4)].width = 12

# Freeze VP column and header
ws_sum.freeze_panes = "B4"

# Sort instruction note below data
note_row = data_row_start + len(all_vps) + 2
ws_sum.cell(row=note_row, column=1,
            value="💡 To sort: select row 3 header → Data → Filter → "
                  "sort TOTAL_FLAGS column Z→A to see most-flagged participants first"
           ).font = Font(italic=True, name="Arial", size=8, color="666666")
ws_sum.merge_cells(start_row=note_row, start_column=1,
                   end_row=note_row, end_column=len(sum_header))

print(f"  Sheet 'FLAG_SUMMARY': {len(all_vps)} participants")

# ── LEGEND ────────────────────────────────────────────────────────────────────
leg = wb.create_sheet(title="LEGEND")
leg.column_dimensions["A"].width = 20
leg.column_dimensions["B"].width = 70
rows_leg = [
    ("COLOUR LEGEND",       ""),
    ("Green",               "All criteria within bounds — clean data point"),
    ("Yellow",              "Outside ± 1 SD from group mean — monitor"),
    ("Orange",              "Outside ± 2 SD from group mean — investigate"),
    ("Red",                 "Physiological range violation OR ± 3 SD OR RSP invalid — hard exclude"),
    ("Grey",                "Missing — excluded in source pipeline (e.g. temp sensor detached)"),
    ("",""),
    ("COLUMN DEFINITIONS",  ""),
    ("raw_value",           "Actual measured value for this subject / session / phase"),
    ("phys_flag",           "Outside defined physiological plausibility range (hard exclude)"),
    ("extremely_low",       "Below group mean − 3 SD"),
    ("extremely_high",      "Above group mean + 3 SD"),
    ("flag_1SD",            "Outside group mean ± 1 SD"),
    ("flag_2SD",            "Outside group mean ± 2 SD"),
    ("flag_3SD",            "Outside group mean ± 3 SD (same as extremely low/high combined)"),
    ("rsp_invalid",         "Breathing frequency outside 9–24 br/min → invalidates HFn"),
    ("n_flags",             "Total number of YES flags across all criteria for this cell"),
    ("exclude_any",         "YES if phys_flag=YES or flag_3SD=YES or rsp_invalid=YES"),
    ("note",                "Plain-text reason summary"),
    ("",""),
    ("TEMPERATURE NOTE",    ""),
    ("Missing temp values", "temp_excluded=1 in source file = sensor was detached or invalid. "
                            "These are NOT code errors — they were already excluded upstream."),
    ("Low temp values",     "Values like 4°C, 8°C, 24°C were caused by the column detector "
                            "accidentally picking up 'temp_excluded' or 'edatemp_dur_min' columns. "
                            "Fixed in v2 by using exact column name 'temp_mean'."),
    ("",""),
    ("FLAG SUMMARY SHEET",  ""),
    ("TOTAL_FLAGS",         "Sum of hard-exclude flags (exclude_any=YES) across all 6 measures"),
    ("RANK",                "1 = fewest flags (cleanest), highest number = most problematic"),
    ("VERDICT",             "CLEAN=0 flags | MONITOR=1-2 | INVESTIGATE=3-5 | REVIEW=>5"),
    ("Sort tip",            "Use Data → Filter on row 3, then sort TOTAL_FLAGS descending"),
]
LMAP = {"Green":FILL_GREEN,"Yellow":FILL_YELLOW,"Orange":FILL_ORANGE,
        "Red":FILL_RED,"Grey":FILL_GREY}
SECTION_HEADERS = ("COLOUR LEGEND","COLUMN DEFINITIONS",
                   "TEMPERATURE NOTE","FLAG SUMMARY SHEET")
for i, (k, v) in enumerate(rows_leg, 1):
    leg.cell(row=i, column=1).value = k
    leg.cell(row=i, column=2).value = v
    leg.cell(row=i, column=1).font  = FB
    leg.cell(row=i, column=2).font  = FN
    leg.cell(row=i, column=2).alignment = Alignment(wrap_text=True)
    if k in LMAP:
        leg.cell(row=i, column=1).fill = LMAP[k]
    if k in SECTION_HEADERS:
        leg.cell(row=i, column=1).font = FW
        leg.cell(row=i, column=1).fill = FILL_HEADER
        leg.merge_cells(start_row=i, start_column=1,
                        end_row=i, end_column=2)
    leg.row_dimensions[i].height = 18

wb.save(OUT_FILE)
print(f"\nSaved → {OUT_FILE}")
print("Sheets: LEGEND + FLAG_SUMMARY + RMSSD + HFn + EDA + Temperature + BF + HR")

  Sheet 'RMSSD': 168 data rows (gm=43.00, gs=20.93)
  Sheet 'HFn': 168 data rows (gm=0.34, gs=0.19)
  Sheet 'EDA': 160 data rows (gm=4.78, gs=8.84)
  Sheet 'Temperature': 160 data rows (gm=32.12, gs=2.75)
  Sheet 'BF': 160 data rows (gm=14.80, gs=3.60)
  Sheet 'HR': 160 data rows (gm=73.55, gs=12.02)
  Sheet 'FLAG_SUMMARY': 21 participants

Saved → C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data\derivatives\outlier-analysis\exclusion_criteria_workbook.xlsx
Sheets: LEGEND + FLAG_SUMMARY + RMSSD + HFn + EDA + Temperature + BF + HR
